In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Membaca data
df = pd.read_csv('wbc.csv')

# 1. Menghapus variabel yang tidak digunakan
df = df.drop(columns=['id', 'Unnamed: 32'])

# 2. Encoding diagnosis
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

# Memisahkan variabel independen dan target
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

print("Jumlah fitur:", X.shape[1])
print("Nama fitur:")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

results = []

for k in [5, 10, 15, 20, 25, 30]:

    pipeline = Pipeline([
        ('selector', SelectKBest(score_func=chi2, k=k)),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        'Jumlah Fitur': k,
        'Accuracy': accuracy
    })

results_df = pd.DataFrame(results)

print(results_df)

# mencari hasil terbaik
best_result = results_df.loc[results_df['Accuracy'].idxmax()]

print("Jumlah fitur terbaik:", best_result['Jumlah Fitur'])
print("Accuracy terbaik:", best_result['Accuracy'])

best_k = int(best_result['Jumlah Fitur'])

best_pipeline = Pipeline([
    ('selector', SelectKBest(score_func=chi2, k=best_k)),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

best_pipeline.fit(X_train, y_train)

selector = best_pipeline.named_steps['selector']

selected_features = X.columns[selector.get_support()]

print("Fitur yang terpilih:")
print(selected_features.tolist())

# evaluasi model
y_pred = best_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

Jumlah fitur: 30
Nama fitur:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']
   Jumlah Fitur  Accuracy
0             5  0.921053
1            10  0.929825
2            15  0.964912
3            20  0.973684
4            25  0.973684
5            30  0.964912
Jumlah fitur terbaik: 20.0
Accuracy terbaik: 0.9736842105263158
Fitur yang terpilih:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'area